# Notebook 06 — GPBoost para clasificación de humedales en el río Bita

Este notebook aplica **GPBoost** a la clasificación de humedales en la cuenca del río Bita y zonas aledañas de la Orinoquía colombiana. El conjunto de datos contiene 479 puntos de referencia con variables extraídas en Google Earth Engine para 2021: Sentinel-2, Sentinel-1, ALOS PALSAR-2 y Copernicus DEM.

El flujo se organiza en dos partes. Primero se aborda el problema binario **humedal / no humedal**, que permite usar el componente espacial gaussiano de GPBoost con una verosimilitud logística. Luego se plantea una extensión multiclase mediante un esquema **One-vs-Rest**, útil cuando se quieren distinguir categorías ecológicas más finas sin perder el modelado explícito de dependencia espacial.

La finalidad del notebook no es producir un mapa final operacional, sino evaluar si GPBoost aporta valor frente a un baseline de Random Forest con coordenadas.

## Compatibilidad con Google Colab

El bloque siguiente instala dependencias solo cuando el notebook se ejecuta en Google Colab. En ejecución local o dentro del repositorio, se asume que el ambiente ya contiene las librerías necesarias.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "gpboost", "shap"],
        check=True,
    )

    repo_dir = Path("/content/repo")
    if not repo_dir.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "-q",
                "https://github.com/linaq11/examen-2-gpboost.git",
                str(repo_dir),
            ],
            check=True,
        )

    os.chdir(repo_dir)
else:
    current_dir = Path.cwd()
    if current_dir.name == "notebooks":
        os.chdir(current_dir.parent)

print(f"Directorio de trabajo: {Path.cwd()}")

## 1. Configuración general

In [ ]:
import json
import warnings
from pathlib import Path

import gpboost as gpb
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

warnings.filterwarnings("ignore", category=UserWarning)

RANDOM_SEED = 1

DATA_DIR = Path("data")
FIGURE_DIR = Path("figuras")
MODEL_DIR = Path("modelos")

FIGURE_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(RANDOM_SEED)

## 2. Carga del dataset extraído desde Google Earth Engine

El archivo base proviene de la extracción de variables realizada en Google Earth Engine sobre los 479 puntos de referencia. Se usa 2021 porque ese año mantiene una cobertura Sentinel-1 adecuada sobre el área de estudio, antes de la pérdida operativa de Sentinel-1B a finales de 2021.

In [ ]:
DATA_CANDIDATES = [
    DATA_DIR / "bita_with_features.csv",
    DATA_DIR / "Bita_features_479_2021_S2_S1_PALSAR_DEM.csv",
    Path("Bita_features_479_2021_S2_S1_PALSAR_DEM.csv"),
]

data_path = next((path for path in DATA_CANDIDATES if path.exists()), None)

if data_path is None:
    expected_files = "\n".join(f"- {path}" for path in DATA_CANDIDATES)
    raise FileNotFoundError(
        "No se encontró el archivo de entrada. Ubica el CSV en una de estas rutas:\n"
        f"{expected_files}"
    )

df = pd.read_csv(data_path)

print(f"Archivo cargado: {data_path}")
print(f"Filas iniciales: {len(df)}")
print(f"Columnas: {len(df.columns)}")

df.head()

## 3. Limpieza y selección de variables

Se conservan 17 predictores derivados de reflectancia óptica, índices espectrales, retrodispersión SAR, PALSAR banda L y elevación. La pendiente se excluye porque puede presentar valores faltantes sobre superficies de agua y no es necesaria para la comparación principal.

Los valores faltantes en `HH_L` y `HV_L`, cuando existen, se imputan con la mediana de su respectiva clase binaria. Si una clase no tuviera mediana disponible, se usa la mediana global como respaldo.

In [ ]:
FEATURE_COLUMNS = [
    "B2", "B3", "B4", "B8", "B11", "B12",
    "NDVI", "NDWI", "MNDWI", "EVI",
    "VV", "VH", "VHdVV", "VHmVV",
    "HH_L", "HV_L",
    "elev",
]

COORD_COLUMNS = ["lon", "lat"]
TARGET_BINARY = "humedal_binario"
TARGET_MULTI = "clase_macro"

required_columns = FEATURE_COLUMNS + COORD_COLUMNS + [TARGET_BINARY, TARGET_MULTI]
missing_columns = sorted(set(required_columns) - set(df.columns))

if missing_columns:
    raise ValueError(
        "El CSV no contiene todas las columnas requeridas. "
        f"Columnas faltantes: {missing_columns}"
    )

df = df.copy()

for column in ["HH_L", "HV_L"]:
    global_median = df[column].median()
    df[column] = df.groupby(TARGET_BINARY)[column].transform(
        lambda values: values.fillna(values.median())
    )
    df[column] = df[column].fillna(global_median)

df = df.dropna(subset=FEATURE_COLUMNS + COORD_COLUMNS + [TARGET_BINARY, TARGET_MULTI])
df = df.reset_index(drop=True)

df[TARGET_BINARY] = pd.to_numeric(df[TARGET_BINARY], errors="raise").astype(int)

print(f"Filas tras limpieza: {len(df)}")
print(f"Distribución de {TARGET_BINARY}:")
print(df[TARGET_BINARY].value_counts().sort_index().to_dict())
print(f"Proporción de humedales: {df[TARGET_BINARY].mean() * 100:.1f}%")

## 4. Normalización de predictores y coordenadas

In [ ]:
feature_scaler = MinMaxScaler()
coord_scaler = MinMaxScaler()

X = feature_scaler.fit_transform(df[FEATURE_COLUMNS])
coords = coord_scaler.fit_transform(df[COORD_COLUMNS])
y = df[TARGET_BINARY].to_numpy(dtype=int)

print(f"X: {X.shape}")
print(f"Coordenadas: {coords.shape}")
print(f"y: {y.shape}")

## 5. División estratificada train/test

Se reserva 25 % del conjunto como test. La partición se estratifica por la clase binaria para conservar una proporción similar de humedales y no humedales en entrenamiento y prueba.

In [ ]:
indices = np.arange(len(y))

idx_train, idx_test = train_test_split(
    indices,
    test_size=0.25,
    stratify=y,
    random_state=RANDOM_SEED,
)

X_train, X_test = X[idx_train], X[idx_test]
y_train, y_test = y[idx_train], y[idx_test]
coords_train, coords_test = coords[idx_train], coords[idx_test]

print(f"Train: {len(y_train)} observaciones ({y_train.mean() * 100:.1f}% humedales)")
print(f"Test:  {len(y_test)} observaciones ({y_test.mean() * 100:.1f}% humedales)")

## 6. Entrenamiento de GPBoost binario

El modelo usa `objective="binary"` y una covarianza exponencial sobre coordenadas normalizadas. Los hiperparámetros del booster se mantienen conservadores para evitar que los árboles absorban toda la estructura espacial que debería ser capturada por el proceso gaussiano.

In [ ]:
gp_model = gpb.GPModel(
    gp_coords=coords_train,
    cov_function="exponential",
    likelihood="bernoulli_logit",
)

train_data = gpb.Dataset(
    X_train,
    y_train,
    feature_name=FEATURE_COLUMNS,
)

gpboost_params = {
    "objective": "binary",
    "metric": "binary_logloss",
    "learning_rate": 0.05,
    "max_depth": 5,
    "min_data_in_leaf": 5,
    "num_leaves": 64,
    "verbose": -1,
}

gpboost_model = gpb.train(
    params=gpboost_params,
    train_set=train_data,
    gp_model=gp_model,
    num_boost_round=300,
)

model_path = MODEL_DIR / "gpboost_bita_binario.json"
gpboost_model.save_model(str(model_path))

print(f"Modelo guardado en: {model_path}")

## 7. Predicción y métricas del modelo binario

In [ ]:
def extract_gpboost_probabilities(prediction):
    """Extract response probabilities from GPBoost predictions."""
    if isinstance(prediction, dict):
        if "response_mean" in prediction:
            return np.asarray(prediction["response_mean"]).ravel()
        return np.asarray(next(iter(prediction.values()))).ravel()

    return np.asarray(prediction).ravel()


def safe_roc_auc(y_true, y_score):
    """Return ROC-AUC when both classes are present; otherwise return NaN."""
    if len(np.unique(y_true)) < 2:
        return np.nan
    return roc_auc_score(y_true, y_score)


gpboost_prediction = gpboost_model.predict(
    data=X_test,
    gp_coords_pred=coords_test,
    predict_var=False,
)

gpboost_prob = extract_gpboost_probabilities(gpboost_prediction)
gpboost_class = (gpboost_prob >= 0.5).astype(int)

gpboost_metrics = {
    "Accuracy": accuracy_score(y_test, gpboost_class),
    "Balanced accuracy": balanced_accuracy_score(y_test, gpboost_class),
    "Kappa": cohen_kappa_score(y_test, gpboost_class),
    "F1": f1_score(y_test, gpboost_class),
    "ROC-AUC": safe_roc_auc(y_test, gpboost_prob),
}

print(f"GPBoost binario — Test (n={len(y_test)}):")
for metric, value in gpboost_metrics.items():
    print(f"  {metric:<18} {value:.3f}")

print("\nReporte de clasificación:")
print(
    classification_report(
        y_test,
        gpboost_class,
        target_names=["No humedal", "Humedal"],
        zero_division=0,
    )
)

## 8. Baseline Random Forest con coordenadas

El baseline usa Random Forest con las mismas variables predictoras y las coordenadas normalizadas como predictores adicionales. Esta comparación permite separar parcialmente el efecto de incluir localización espacial como covariable frente a modelar explícitamente dependencia espacial mediante GPBoost.

In [ ]:
X_train_rf = np.column_stack([X_train, coords_train])
X_test_rf = np.column_stack([X_test, coords_test])

rf_model = RandomForestClassifier(
    n_estimators=500,
    random_state=RANDOM_SEED,
    n_jobs=-1,
)

rf_model.fit(X_train_rf, y_train)

rf_prob = rf_model.predict_proba(X_test_rf)[:, 1]
rf_class = (rf_prob >= 0.5).astype(int)

rf_metrics = {
    "Accuracy": accuracy_score(y_test, rf_class),
    "Balanced accuracy": balanced_accuracy_score(y_test, rf_class),
    "Kappa": cohen_kappa_score(y_test, rf_class),
    "F1": f1_score(y_test, rf_class),
    "ROC-AUC": safe_roc_auc(y_test, rf_prob),
}

print("Random Forest + coordenadas:")
for metric, value in rf_metrics.items():
    print(f"  {metric:<18} {value:.3f}")

## 9. Tabla comparativa y matriz de confusión

In [ ]:
comparison_table = pd.DataFrame(
    [
        {"Método": "GPBoost (boost + GP)", **gpboost_metrics},
        {"Método": "Random Forest + coords", **rf_metrics},
    ]
).round(3)

comparison_path = DATA_DIR / "bita_comparativa_metodos.csv"
comparison_table.to_csv(comparison_path, index=False)

comparison_table

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

for ax, (name, prediction) in zip(
    axes,
    [
        ("GPBoost", gpboost_class),
        ("Random Forest", rf_class),
    ],
):
    cm = confusion_matrix(y_test, prediction)
    image = ax.imshow(cm, cmap="Blues")

    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(["No humedal", "Humedal"])
    ax.set_yticklabels(["No humedal", "Humedal"])
    ax.set_xlabel("Predicho")
    ax.set_ylabel("Real")
    ax.set_title(name)

    threshold = cm.max() / 2
    for row in range(cm.shape[0]):
        for col in range(cm.shape[1]):
            ax.text(
                col,
                row,
                cm[row, col],
                ha="center",
                va="center",
                color="white" if cm[row, col] > threshold else "black",
                fontsize=14,
                fontweight="bold",
            )

    plt.colorbar(image, ax=ax, fraction=0.046)

plt.tight_layout()

figure_path = FIGURE_DIR / "fig9_bita_confusion.png"
plt.savefig(figure_path, dpi=150, bbox_inches="tight")
plt.show()

## 10. Distribución espacial de predicciones

La figura compara las etiquetas reales del conjunto de test con las predicciones binarias de GPBoost. Esta visualización es exploratoria: permite identificar si los errores se concentran en sectores particulares del espacio, pero no reemplaza una validación espacial independiente.

In [ ]:
test_lon = df.iloc[idx_test]["lon"].to_numpy()
test_lat = df.iloc[idx_test]["lat"].to_numpy()

fig, axes = plt.subplots(1, 2, figsize=(13, 6))

panels = [
    (axes[0], y_test, f"Real (test, n={len(y_test)})"),
    (axes[1], gpboost_class, "Predicho GPBoost"),
]

for ax, values, title in panels:
    point_colors = ["#d95f02" if value == 0 else "#1b9e77" for value in values]

    ax.scatter(
        test_lon,
        test_lat,
        c=point_colors,
        s=40,
        edgecolor="k",
        linewidth=0.4,
    )

    ax.set_title(title)
    ax.set_xlabel("Longitud")
    ax.set_ylabel("Latitud")
    ax.grid(alpha=0.3)

legend_handles = [
    mpatches.Patch(color="#d95f02", label="No humedal"),
    mpatches.Patch(color="#1b9e77", label="Humedal"),
]

fig.legend(
    handles=legend_handles,
    loc="lower center",
    ncol=2,
    bbox_to_anchor=(0.5, -0.02),
)

plt.tight_layout()

figure_path = FIGURE_DIR / "fig10_bita_mapa.png"
plt.savefig(figure_path, dpi=150, bbox_inches="tight")
plt.show()

## 11. Importancia de variables

In [ ]:
feature_importance = gpboost_model.feature_importance(importance_type="gain")

importance_table = (
    pd.DataFrame(
        {
            "feature": FEATURE_COLUMNS,
            "importance": feature_importance,
        }
    )
    .sort_values("importance")
    .reset_index(drop=True)
)

fig, ax = plt.subplots(figsize=(8, 6))

ax.barh(importance_table["feature"], importance_table["importance"])
ax.set_xlabel("Importancia por ganancia")
ax.set_title("Importancia de variables — GPBoost binario")

plt.tight_layout()

figure_path = FIGURE_DIR / "fig11_bita_feature_importance.png"
plt.savefig(figure_path, dpi=150, bbox_inches="tight")
plt.show()

importance_table.tail(8)

## 12. Extensión multiclase mediante One-vs-Rest

La clasificación binaria simplifica el problema para aprovechar la verosimilitud logística con componente espacial. Sin embargo, el campo `clase_macro` distingue categorías ecológicamente más específicas: por ejemplo, no humedal, agua abierta permanente, humedal vegetado permanente, humedal temporal y humedal potencial.

Como GPBoost tiene restricciones prácticas para combinar `objective="multiclass"` con un `gp_model`, se usa un esquema **One-vs-Rest**. En este enfoque se entrena un clasificador binario por clase y la clase final se asigna con el mayor valor de probabilidad posterior.

Esta estrategia es una aproximación operativa. Las probabilidades de los modelos OvR no están calibradas conjuntamente, por lo que la interpretación principal debe centrarse en la clasificación relativa y no en probabilidades multiclase estrictamente normalizadas.

In [ ]:
label_encoder = LabelEncoder()
y_multi = label_encoder.fit_transform(df[TARGET_MULTI])
class_names = label_encoder.classes_
n_classes = len(class_names)

class_counts = pd.Series(y_multi).value_counts().sort_index()

print(f"Clases macro ({n_classes}):")
for class_id, class_name in enumerate(class_names):
    print(f"  {class_id} → {class_name} ({class_counts[class_id]} puntos)")

if class_counts.min() < 2:
    raise ValueError(
        "Al menos una clase tiene menos de dos observaciones. "
        "No es posible hacer una partición estratificada robusta."
    )

idx_train_multi, idx_test_multi = train_test_split(
    np.arange(len(y_multi)),
    test_size=0.25,
    stratify=y_multi,
    random_state=RANDOM_SEED,
)

X_train_multi, X_test_multi = X[idx_train_multi], X[idx_test_multi]
coords_train_multi, coords_test_multi = coords[idx_train_multi], coords[idx_test_multi]
y_train_multi, y_test_multi = y_multi[idx_train_multi], y_multi[idx_test_multi]

print(f"\nTrain: {len(y_train_multi)} observaciones")
print(f"Test:  {len(y_test_multi)} observaciones")

### 12.1. Entrenamiento de clasificadores binarios

Cada clase se modela como un problema binario independiente: pertenecer o no pertenecer a esa clase. Todos los modelos usan la misma especificación de covarianza y los mismos hiperparámetros del booster.

In [ ]:
ovr_probabilities = np.zeros((len(y_test_multi), n_classes))
ovr_models = {}

ovr_params = {
    "objective": "binary",
    "metric": "binary_logloss",
    "learning_rate": 0.05,
    "max_depth": 5,
    "min_data_in_leaf": 5,
    "num_leaves": 64,
    "verbose": -1,
}

for class_id, class_name in enumerate(class_names):
    y_train_binary = (y_train_multi == class_id).astype(int)
    y_test_binary = (y_test_multi == class_id).astype(int)

    gp_model_class = gpb.GPModel(
        gp_coords=coords_train_multi,
        cov_function="exponential",
        likelihood="bernoulli_logit",
    )

    train_data_class = gpb.Dataset(
        X_train_multi,
        y_train_binary,
        feature_name=FEATURE_COLUMNS,
    )

    model_class = gpb.train(
        params=ovr_params,
        train_set=train_data_class,
        gp_model=gp_model_class,
        num_boost_round=200,
    )

    prediction_class = model_class.predict(
        data=X_test_multi,
        gp_coords_pred=coords_test_multi,
        predict_var=False,
    )

    ovr_probabilities[:, class_id] = extract_gpboost_probabilities(prediction_class)

    class_model_path = MODEL_DIR / f"gpboost_bita_ovr_class_{class_id}.json"
    model_class.save_model(str(class_model_path))
    ovr_models[class_id] = model_class

    auc_value = safe_roc_auc(y_test_binary, ovr_probabilities[:, class_id])
    print(f"Clase {class_id} ({class_name}): ROC-AUC OvR = {auc_value:.3f}")

ovr_class = ovr_probabilities.argmax(axis=1)

print(f"\nPredicciones OvR generadas: argmax sobre {n_classes} probabilidades.")

### 12.2. Métricas multiclase OvR

In [ ]:
ovr_metrics = {
    "Accuracy": accuracy_score(y_test_multi, ovr_class),
    "Balanced accuracy": balanced_accuracy_score(y_test_multi, ovr_class),
    "Kappa": cohen_kappa_score(y_test_multi, ovr_class),
    "F1 macro": f1_score(y_test_multi, ovr_class, average="macro"),
}

print(f"GPBoost OvR ({n_classes} clases) — Test (n={len(y_test_multi)}):")
for metric, value in ovr_metrics.items():
    print(f"  {metric:<18} {value:.3f}")

print("\nReporte de clasificación:")
print(
    classification_report(
        y_test_multi,
        ovr_class,
        target_names=class_names,
        zero_division=0,
    )
)

### 12.3. Baseline Random Forest multiclase con coordenadas

Para contraste se ajusta un Random Forest multiclase nativo con las mismas variables predictoras y coordenadas. A diferencia de GPBoost OvR, este baseline no descompone el problema en clasificadores binarios.

In [ ]:
X_train_multi_rf = np.column_stack([X_train_multi, coords_train_multi])
X_test_multi_rf = np.column_stack([X_test_multi, coords_test_multi])

rf_multi_model = RandomForestClassifier(
    n_estimators=500,
    random_state=RANDOM_SEED,
    n_jobs=-1,
)

rf_multi_model.fit(X_train_multi_rf, y_train_multi)

rf_multi_class = rf_multi_model.predict(X_test_multi_rf)

rf_multi_metrics = {
    "Accuracy": accuracy_score(y_test_multi, rf_multi_class),
    "Balanced accuracy": balanced_accuracy_score(y_test_multi, rf_multi_class),
    "Kappa": cohen_kappa_score(y_test_multi, rf_multi_class),
    "F1 macro": f1_score(y_test_multi, rf_multi_class, average="macro"),
}

print("Random Forest multiclase + coordenadas:")
for metric, value in rf_multi_metrics.items():
    print(f"  {metric:<18} {value:.3f}")

### 12.4. Tabla comparativa multiclase y matriz de confusión

In [ ]:
comparison_multi_table = pd.DataFrame(
    [
        {"Método": "GPBoost OvR (binarios + GP)", **ovr_metrics},
        {"Método": "Random Forest + coords", **rf_multi_metrics},
    ]
).round(3)

comparison_multi_path = DATA_DIR / "bita_comparativa_metodos_multiclase.csv"
comparison_multi_table.to_csv(comparison_multi_path, index=False)

comparison_multi_table

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, (name, prediction) in zip(
    axes,
    [
        ("GPBoost OvR", ovr_class),
        ("Random Forest", rf_multi_class),
    ],
):
    cm = confusion_matrix(y_test_multi, prediction, labels=range(n_classes))
    image = ax.imshow(cm, cmap="Blues")

    ax.set_xticks(range(n_classes))
    ax.set_yticks(range(n_classes))
    ax.set_xticklabels(class_names, rotation=30, ha="right", fontsize=7)
    ax.set_yticklabels(class_names, fontsize=7)
    ax.set_xlabel("Predicho")
    ax.set_ylabel("Real")
    ax.set_title(name)

    threshold = cm.max() / 2
    for row in range(n_classes):
        for col in range(n_classes):
            ax.text(
                col,
                row,
                cm[row, col],
                ha="center",
                va="center",
                color="white" if cm[row, col] > threshold else "black",
                fontsize=8,
            )

    plt.colorbar(image, ax=ax, fraction=0.046)

plt.tight_layout()

figure_path = FIGURE_DIR / "fig12_bita_confusion_multiclase.png"
plt.savefig(figure_path, dpi=150, bbox_inches="tight")
plt.show()

## 13. Lectura final

El modelo binario permite evaluar de manera directa la separación entre humedal y no humedal con un componente espacial explícito. El Random Forest con coordenadas funciona como baseline fuerte, pero no modela la autocorrelación residual con la misma estructura probabilística.

La extensión One-vs-Rest conserva el uso de un GP independiente por clase. Esto resulta útil para explorar categorías ecológicas más finas, aunque sus probabilidades no deben interpretarse como una distribución multiclase plenamente calibrada.

Tres puntos operativos quedan abiertos para trabajo posterior:

1. **Clases minoritarias.** Si `humedal_potencial` u otra categoría tiene baja frecuencia, conviene ampliar el muestreo o incorporar pesos de clase.
2. **Validación espacial.** La partición aleatoria es informativa, pero debería complementarse con validación espacial por bloques hidrográficos o subzonas de la cuenca.
3. **Transferencia espacial.** Una prueba natural consiste en entrenar sobre Bita y evaluar transferencia hacia otra cuenca, como Ariporo, para verificar generalización fuera del área de entrenamiento.